# Chapter 9 — Reuse Before Recompute

## Question

**Two sessions can send similar numbers of tokens. Why might one require much more repeated computation?**

Falsifiable version: do two equal-sized edits at different positions produce different reusable-prefix lengths? If yes, token count alone cannot describe repeated-context economics.

Everything below is prefix geometry over exact rendered identity. No provider cache hit is claimed, no pricing is real, no latency is measured.

## Setup — requests as ordered spans

In [ ]:
SPAN_TOKENS = {'A': 3000, 'B': 2500, 'C': 2000, 'D': 1500, 'E': 1200, 'F': 1200}

def shared_prefix(r1, r2):
    """Longest common leading span sequence over exact span identity."""
    shared = []
    for a, b in zip(r1, r2):
        if a == b:
            shared.append(a)
        else:
            break
    total = sum(SPAN_TOKENS[s] for s in r1)
    reuse = sum(SPAN_TOKENS[s] for s in shared)
    first = (len(shared), r1[len(shared)] if len(shared) < len(r1) else None,
             r2[len(shared)] if len(shared) < len(r2) else None)
    return {'shared': shared, 'reuse_tokens': reuse,
            'first_divergence': first, 'reusable_ratio': reuse / total}

def mutation_radius(request, divergence_index):
    """Tokens downstream of the first divergence: the footprint an edit invalidates."""
    return sum(SPAN_TOKENS[s] for s in request[divergence_index:])

R1 = ['A', 'B', 'C', 'D', 'E']
R2 = ['A', 'B', 'C', 'D', 'F']
print('spans:', {s: SPAN_TOKENS[s] for s in R1})

## Baseline — the reusable prefix

In [ ]:
g = shared_prefix(R1, R2)
print(f"R1 = {' '.join(R1)}; R2 = {' '.join(R2)}")
print(f"shared = {' '.join(g['shared'])} ({g['reuse_tokens']} tokens)")
print(f'first divergence at index {g["first_divergence"][0]}: {g["first_divergence"][1]} vs {g["first_divergence"][2]}')
print(f'reusable-prefix ratio: {g["reusable_ratio"]:.1%}')
assert g['shared'] == ['A', 'B', 'C', 'D']
assert g['reuse_tokens'] == 9000

## Intervention — early versus late mutation

Four conditions with identical semantics (mutations touch inert fields only): stable append-only; a tiny early edit; the same-sized edit placed late; a historical rewrite. Edit size is matched between B and C; footprints are not.

In [ ]:
BASE = ['A', 'B', 'C', 'D', 'E']
INERT = {'B_early': 100, 'D_late': 100}  # same-sized inert edits, different positions
SPAN2 = dict(SPAN_TOKENS)
SPAN2.update({'B*': SPAN_TOKENS['B'], 'D*': SPAN_TOKENS['D']})

cond_b = ['A', 'B*', 'C', 'D', 'E']   # 100-token inert change near the start
cond_c = ['A', 'B', 'C', 'D*', 'E']    # 100-token inert change near the end
cond_d = ['A', 'SUM', 'E']             # older segment rewritten as one fixed span
SPAN2['SUM'] = SPAN_TOKENS['B'] + SPAN_TOKENS['C'] + SPAN_TOKENS['D']

def geometry(prev, nxt, toks):
    shared = []
    for a, b in zip(prev, nxt):
        if a == b:
            shared.append(a)
        else:
            break
    div = len(shared)
    downstream = sum(toks[s] for s in nxt[div:])
    return div, sum(toks[s] for s in shared), downstream

for name, nxt in [('B early mutation', cond_b), ('C late mutation', cond_c), ('D rewrite', cond_d)]:
    div, reuse, radius = geometry(BASE, nxt, SPAN2)
    print(f'{name:16s} edited=100 tokens first-divergence={div} reusable={reuse} mutation-radius={radius}')
early = geometry(BASE, cond_b, SPAN2)
late = geometry(BASE, cond_c, SPAN2)
assert early[2] > late[2], 'a tiny early edit invalidates more than the same edit placed late'
print('Small edit size is not a small recomputation footprint.')

## Append versus rewrite — a four-turn session

In [ ]:
append_turns = [['A', 'B'], ['A', 'B', 'C'], ['A', 'B', 'C', 'D'], ['A', 'B', 'C', 'D', 'E']]
rewrite_turns = [['A', 'B'], ['A', 'B', 'C'], ['A', 'S1'], ['A', 'S1', 'E']]
SPAN3 = dict(SPAN_TOKENS)
SPAN3['S1'] = SPAN_TOKENS['B'] + SPAN_TOKENS['C']

def progression(turns, toks):
    out = [sum(toks[s] for s in turns[0])]
    for p, n in zip(turns, turns[1:]):
        _, reuse, _ = geometry(p, n, toks)
        out.append(reuse)
    return out

print('append-only reusable prefix by turn: ', progression(append_turns, SPAN_TOKENS))
print('rewrite reusable prefix by turn:     ', progression(rewrite_turns, SPAN3))
assert progression(append_turns, SPAN_TOKENS) == sorted(progression(append_turns, SPAN_TOKENS))
assert progression(rewrite_turns, SPAN3)[2] < progression(append_turns, SPAN_TOKENS)[2]
print('Rewrite buys fewer tokens per turn at the price of orphaned downstream reuse.')

## Synthetic economics — amortisation and the minimum-length trap

SYNTHETIC CACHE ECONOMICS (fictional schedule, not provider pricing): uncached token 1.0, write 1.2, read 0.1 cost units. Minimum reusable prefix 1,000 fixture tokens.

In [ ]:
WRITE, READ, FULL = 1.2, 0.1, 1.0
prefix = 9000  # the A-D reusable prefix
print(f"{'requests':>8s} {'with reuse':>10s} {'uncached':>8s}")
for n in (1, 2, 10):
    with_reuse = prefix * WRITE + (n - 1) * prefix * READ
    plain = n * prefix * FULL
    print(f'{n:8d} {with_reuse:10.0f} {plain:8.0f}')
assert prefix * WRITE + 9 * prefix * READ < 10 * prefix * FULL

MIN_PREFIX = 1000
short = 900
print(f'\n900-token perfectly stable prefix earns reuse: {short >= MIN_PREFIX}')
assert short < MIN_PREFIX
print('Padding with useless content to cross a cache threshold would optimise cache accounting, not context quality.')

## Cacheable is not relevant — independent axes

In [ ]:
stable_irrelevant = {'reuse_geometry': 'excellent', 'deserves_admission': False}
volatile_required = {'reuse_geometry': 'poor', 'deserves_admission': True}
print(f"stable irrelevant item: {stable_irrelevant}")
print(f"volatile required item: {volatile_required}")
assert stable_irrelevant['deserves_admission'] is False
assert volatile_required['deserves_admission'] is True
print('A cached irrelevance is still an irrelevance, only cheaper.')

## Try it

1. Move the inert edit from `B*` to `C` and watch the reusable prefix grow while edit size stays 100.
2. Raise MIN_PREFIX above 9,000: even the A-D prefix earns nothing, and the amortisation table inverts.
3. Extend the append-only session two turns and confirm the progression stays monotonic.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(shared_prefix(['A', 'B', 'C'], ['A', 'B*', 'C']))

## What this demonstrates

- The position of change determines the amount of potentially reusable leading computation: matched 100-token edits produce different mutation radii.
- Token count alone is not a sufficient economic description of repeated context.
- Append-only growth preserves prefixes monotonically; rewrite orphans them.

## What this does not demonstrate

- That any actual provider cache hit occurred, or any real pricing or TTL behaviour.
- Any latency saving.
- That cached content is useful, or that stable content should be retained merely for reuse.

## Connection to the chapter

Every deletion considered next carries two sizes — the material removed and the cached prefix disturbed:

> Every deletion we make next can save context while simultaneously moving the first divergence earlier and destroying reuse.

That is Chapter 10.